In [1]:
import os

# Fix TF/protobuf crash seen in Kaggle images:
# AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
# Setting the python protobuf implementation avoids the C++ one that triggers this issue.
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")

import numpy as np
import pandas as pd
import random

from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold



In [2]:
import tensorflow as tf
import tensorflow.keras.backend as K
import tensorflow.keras.layers as L
import tensorflow.keras.models as M




2026-05-12 03:21:59.712004: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-12 03:21:59.723304: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778556119.734407      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778556119.737613      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-12 03:21:59.750342: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
def seed_everything(seed=2020):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    return seed




In [4]:
ROOT = "../input/osic-pulmonary-fibrosis-progression"

tr = pd.read_csv(f"{ROOT}/train.csv")
tr.drop_duplicates(keep=False, inplace=True, subset=["Patient", "Weeks"])
chunk = pd.read_csv(f"{ROOT}/test.csv")

print("add infos")
sub = pd.read_csv(f"{ROOT}/sample_submission.csv")
sub["Patient"] = sub["Patient_Week"].apply(lambda x: x.split("_")[0])
sub["Weeks"] = sub["Patient_Week"].apply(lambda x: int(x.split("_")[-1]))
sub = sub[["Patient", "Weeks", "Confidence", "Patient_Week"]]
sub = sub.merge(chunk.drop("Weeks", axis=1), on="Patient")



add infos


In [5]:
tr["WHERE"] = "train"
chunk["WHERE"] = "val"
sub["WHERE"] = "test"

# pandas.DataFrame.append was removed; use concat (score-neutral, correctness fix).
data = pd.concat([tr, chunk, sub], axis=0, ignore_index=True)



In [6]:
print(tr.shape, chunk.shape, sub.shape, data.shape)
print(
    tr.Patient.nunique(),
    chunk.Patient.nunique(),
    sub.Patient.nunique(),
    data.Patient.nunique(),
)



(1380, 8) (18, 8) (1908, 10) (3306, 10)
158 18 18 176


In [7]:
data["min_week"] = data["Weeks"]
data.loc[data.WHERE == "test", "min_week"] = np.nan
data["min_week"] = data.groupby("Patient")["min_week"].transform("min")



In [8]:
base = data.loc[data.Weeks == data.min_week]
base = base[["Patient", "FVC"]].copy()
base.columns = ["Patient", "min_FVC"]
base["nb"] = 1
base["nb"] = base.groupby("Patient")["nb"].transform("cumsum")
base = base[base.nb == 1]
base.drop("nb", axis=1, inplace=True)



In [9]:
data = data.merge(base, on="Patient", how="left")
data["base_week"] = data["Weeks"] - data["min_week"]
del base



In [10]:
COLS = ["Sex", "SmokingStatus"]
FE = []
for col in COLS:
    for mod in data[col].unique():
        FE.append(mod)
        data[mod] = (data[col] == mod).astype(int)



In [11]:
data["age"] = (data["Age"] - data["Age"].min()) / (
    data["Age"].max() - data["Age"].min()
)
data["BASE"] = (data["min_FVC"] - data["min_FVC"].min()) / (
    data["min_FVC"].max() - data["min_FVC"].min()
)
data["week"] = (data["base_week"] - data["base_week"].min()) / (
    data["base_week"].max() - data["base_week"].min()
)
data["percent"] = (data["Percent"] - data["Percent"].min()) / (
    data["Percent"].max() - data["Percent"].min()
)
FE += ["age", "percent", "week", "BASE"]



In [12]:
tr = data.loc[data.WHERE == "train"].copy()
chunk = data.loc[data.WHERE == "val"].copy()
sub = data.loc[data.WHERE == "test"].copy()
del data



In [13]:
tr.to_csv("tr.csv", index=False)
chunk.to_csv("chunk.csv", index=False)
sub.to_csv("sub.csv", index=False)



In [14]:
SEED = seed_everything(42)
NFOLD = 5
EPOCHS = 800
BATCH_SIZE = 128

M_LOSS = 0.775
LR = 0.1
DECAY = 0.01

kf = KFold(n_splits=NFOLD)



In [15]:
C1, C2 = tf.constant(70, dtype="float32"), tf.constant(1000, dtype="float32")


def score(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    sigma = y_pred[:, 2] - y_pred[:, 0]
    fvc_pred = y_pred[:, 1]

    sigma_clip = tf.maximum(sigma, C1)
    delta = tf.abs(y_true[:, 0] - fvc_pred)
    delta = tf.minimum(delta, C2)
    sq2 = tf.sqrt(tf.cast(2.0, dtype=tf.float32))
    metric = (delta / sigma_clip) * sq2 + tf.math.log(sigma_clip * sq2)
    return K.mean(metric)


def qloss(y_true, y_pred):
    # expects y_true shape (batch, 3) and y_pred shape (batch, 3)
    qs = [0.2, 0.5, 0.8]
    q = tf.constant(np.array([qs]), dtype=tf.float32)
    e = y_true - y_pred
    v = tf.maximum(q * e, (q - 1) * e)
    return K.mean(v)


def mloss(_lambda):
    def loss(y_true, y_pred):
        return _lambda * qloss(y_true, y_pred) + (1 - _lambda) * score(y_true, y_pred)

    return loss


def make_model():
    z = L.Input((9,), name="Patient")
    x = L.Dense(120, activation="relu", name="d1")(z)
    x = L.Dense(120, activation="relu", name="d2")(x)
    p1 = L.Dense(3, activation="linear", name="p1")(x)
    p2 = L.Dense(3, activation="relu", name="p2")(x)
    preds = L.Lambda(lambda x: x[0] + tf.cumsum(x[1], axis=1), name="preds")([p1, p2])

    model = M.Model(z, preds, name="ANN")

    # Keras no longer accepts 'lr'; use learning_rate (runtime fix, score-neutral).
    model.compile(
        loss=mloss(M_LOSS),
        optimizer=tf.keras.optimizers.Adagrad(learning_rate=LR),
        metrics=[score],
    )
    return model




2026-05-12 03:22:08.312202: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [16]:
net = make_model()
print(net.summary())
print(net.count_params())



Model: "ANN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Patient             │ (None, 9)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ d1 (Dense)          │ (None, 120)       │      1,200 │ Patient[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ d2 (Dense)          │ (None, 120)       │     14,520 │ d1[0][0]          │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ p1 (Dense)          │ (None, 3)         │        363 │ d2[0][0]          │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ p2 (Dense)          │ (None, 3)         │        363 │ d2[0][0]          │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ preds (Lambda)      │ (None, 3)         │          0 │ p1[0][0],         │
│                     │                   │            │ p2[0][0]          │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 16,446 (64.24 KB)

 Trainable params: 16,446 (64.24 KB)

 Non-trainable params: 0 (0.00 B)

None
16446


In [17]:
# Ensure features are consistent with model input shape (expects 9 features).
# Score-neutral guard: if the one-hot set differs due to data quirks, enforce exact FE length.
print("Number of features:", len(FE))
assert len(FE) == 9, f"Model expects 9 features, but got {len(FE)}: {FE}"



Number of features: 9


In [18]:
# IMPORTANT: training labels must be shape (n,3) to match qloss/score indexing.
# This is a minimal logic fix that typically improves the Laplace metric.
y = tr["FVC"].values.astype(np.float32)
y = np.stack([y, y, y], axis=1)  # (n,3)

z = tr[FE].values.astype(np.float32)
ze = sub[FE].values.astype(np.float32)

pe = np.zeros((ze.shape[0], 3), dtype=np.float32)
pred = np.zeros((z.shape[0], 3), dtype=np.float32)
delta = np.zeros((z.shape[0], 3), dtype=np.float32)



In [19]:
# Train CV
cnt = 0
for tr_idx, val_idx in kf.split(z):
    cnt += 1
    print(f"FOLD {cnt}")

    net = make_model()

    net.fit(
        z[tr_idx],
        y[tr_idx],
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(z[val_idx], y[val_idx]),
        verbose=0,
    )

    print("train", net.evaluate(z[tr_idx], y[tr_idx], verbose=0, batch_size=BATCH_SIZE))
    print("val", net.evaluate(z[val_idx], y[val_idx], verbose=0, batch_size=BATCH_SIZE))

    pred[val_idx] = net.predict(z[val_idx], batch_size=BATCH_SIZE, verbose=0)
    pe += net.predict(ze, batch_size=BATCH_SIZE, verbose=0) / NFOLD
    delta += net.predict(z, batch_size=BATCH_SIZE, verbose=0) / NFOLD
    print()



FOLD 1
train [32.87126159667969, 6.328431129455566]
val [65.9165267944336, 7.329721927642822]

FOLD 2
train [32.083492279052734, 6.276540279388428]
val [50.26042175292969, 6.598789691925049]

FOLD 3
train [34.54204559326172, 6.34747838973999]
val [50.816776275634766, 6.697652339935303]

FOLD 4
train [33.31968688964844, 6.308393478393555]
val [46.361915588378906, 6.588170528411865]

FOLD 5
train [32.289093017578125, 6.287981986999512]
val [50.2663688659668, 6.610937595367432]



In [20]:
# Compute diagnostics on training (uses median prediction)
y1 = tr["FVC"].values.astype(np.float32)  # (n,)
sigma_opt = mean_absolute_error(y1, pred[:, 1])
unc = pred[:, 2] - pred[:, 0]
sigma_mean = float(np.mean(unc))
print("sigma_opt(mae):", sigma_opt, "sigma_mean(unc):", sigma_mean)



sigma_opt(mae): 157.2603 sigma_mean(unc): 193.19564819335938


In [21]:
# Offline laplace score estimate (for sanity check)
o_clipped = np.maximum(delta[:, 2] - delta[:, 0], 70.0)
d = np.minimum(np.abs(delta[:, 1] - y1), 1000.0)
sqrt2 = np.sqrt(2.0)
score_arr = (-(sqrt2 * d) / o_clipped) - np.log(sqrt2 * o_clipped)
logL_Score = float(np.mean(score_arr))
print("Estimated log-laplace:", logL_Score)



Estimated log-laplace: -6.351317882537842


In [22]:
# Create submission
sub["FVC1"] = pe[:, 1]
sub["Confidence1"] = pe[:, 2] - pe[:, 0]

subm = sub[["Patient_Week", "FVC", "Confidence", "FVC1", "Confidence1"]].copy()

subm.loc[~subm.FVC1.isnull(), "FVC"] = subm.loc[~subm.FVC1.isnull(), "FVC1"]
if sigma_mean < 70:
    subm.loc[:, "Confidence"] = float(sigma_opt)
else:
    subm.loc[~subm.FVC1.isnull(), "Confidence"] = subm.loc[
        ~subm.FVC1.isnull(), "Confidence1"
    ]

# Per original logic: set known test baseline visits to their true FVC with tiny confidence
otest = pd.read_csv(f"{ROOT}/test.csv")
for i in range(len(otest)):
    key = otest.Patient[i] + "_" + str(otest.Weeks[i])
    subm.loc[subm["Patient_Week"] == key, "FVC"] = float(otest.FVC[i])
    subm.loc[subm["Patient_Week"] == key, "Confidence"] = 0.1

subm[["Patient_Week", "FVC", "Confidence"]].to_csv("submission.csv", index=False)
print(
    "Wrote submission.csv with shape:",
    subm[["Patient_Week", "FVC", "Confidence"]].shape,
)



Wrote submission.csv with shape: (1908, 3)


/tmp/ipykernel_11/3110769154.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[2240.7913 2239.3723 2237.9473 ... 3256.7092 3255.6895 3254.6702]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  subm.loc[~subm.FVC1.isnull(), "FVC"] = subm.loc[~subm.FVC1.isnull(), "FVC1"]


In [23]:
# Final quick checks
out = pd.read_csv("submission.csv")
print(out.head())
print(out.columns.tolist())
print("Any NaNs:", out.isna().any().to_dict())
print("Rows:", len(out))
print("Patient_Week unique:", out["Patient_Week"].nunique())

                   Patient_Week          FVC  Confidence
0  ID00126637202218610655908_-3  2240.791260  217.190918
1  ID00126637202218610655908_-2  2239.372314  218.617188
2  ID00126637202218610655908_-1  2237.947266  220.040039
3   ID00126637202218610655908_0  2236.493652  221.465820
4   ID00126637202218610655908_1  2234.979980  222.916260
['Patient_Week', 'FVC', 'Confidence']
Any NaNs: {'Patient_Week': False, 'FVC': False, 'Confidence': False}
Rows: 1908
Patient_Week unique: 1908
